# MCD-rPPG Dataset: Processing with Statistics

This notebook processes the MCD-rPPG dataset and provides comprehensive statistics at the end.

## Features
- Process 10 video files or full dataset
- Extract 9 facial ROIs using MediaPipe
- Chunk videos into 450-frame segments with 150-frame overlap
- Synchronize with PPG signals
- Display detailed statistics at the end

## Configuration
Set PROCESSING_MODE = 'test' for 10 files or 'full' for all dataset.

In [ ]:
!pip install imageio[ffmpeg] mediapipe scikit-video opencv-python numba scipy tqdm -q

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import imageio
import cv2
from tqdm import tqdm
from datetime import datetime
import warnings
import gc
import time
import json

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8')

DATASET_PATH = '/home/cristic/data/Bgeorge/mcd_rppg/snapshots/929fb19c5ff2b5c8ed64a7c3a123744346674e88/'
DB_PATH = os.path.join(DATASET_PATH, 'db.csv')
MEDIAPIPE_MODEL_PATH = '/home/cristic/face_landmarker.task'
OUTPUT_PATH = '/home/cristic/preprocessed_data'

CHUNK_SIZE = 450
OVERLAP_SIZE = 150
ROI_BOX_SIZE = (24, 24)
SMOOTHING_WINDOW = 5

PROCESSING_MODE = 'test'
NUM_TEST_FILES = 10

ROIS = {
    'full_face': list(range(468)),
    'forehead': [10, 67, 69, 108, 109, 151, 337, 338, 297, 299, 9, 8],
    'left_eye': [33, 7, 163, 144, 145, 153, 154, 155, 133, 173, 157, 158, 159, 160, 161, 246],
    'right_eye': [362, 382, 381, 380, 374, 373, 390, 249, 263, 466, 388, 387, 386, 385, 384, 398],
    'nose': [1, 2, 98, 327, 328, 2, 4, 5, 195, 197, 6, 168],
    'mouth': [61, 146, 91, 181, 84, 17, 314, 405, 321, 375, 291, 308, 324, 318, 402, 317, 14, 87, 178, 95],
    'chin': [152, 148, 176, 149, 150, 136, 172, 377, 400, 378, 379, 365, 397],
    'right_cheek_50': [50],
    'left_cheek_280': [280],
    'chin_199': [199]
}

CHUNKS_PATH = os.path.join(OUTPUT_PATH, 'chunks')
os.makedirs(CHUNKS_PATH, exist_ok=True)
print('Configuration loaded')

In [ ]:
print('Loading database...')
if os.path.exists(DB_PATH):
    df = pd.read_csv(DB_PATH)
    meta_df = df.copy()
    print(f'Loaded {len(meta_df)} entries')
else:
    print(f'DB not found at {DB_PATH}')
    meta_df = pd.DataFrame()

file_columns = ['ecg', 'video', 'meta', 'ppg_sync']
for col in file_columns:
    if col in meta_df.columns:
        meta_df[f'{col}_full'] = meta_df[col].apply(
            lambda x: os.path.join(DATASET_PATH, x) if not os.path.isabs(x) else x
        )

meta_df['subject_id'] = meta_df.get('patient_id', meta_df.get('subject_id', 'unknown'))
meta_df['condition'] = meta_df.get('step', meta_df.get('condition', 'unknown'))
meta_df['camera_type'] = meta_df.get('camera', meta_df.get('camera_type', 'unknown'))
meta_df['view_type'] = meta_df.get('view', meta_df.get('view_type', 'unknown'))

valid_df = meta_df.dropna(subset=['video_full', 'ppg_sync_full'])
print(f'Valid entries: {len(valid_df)}')

if PROCESSING_MODE == 'test':
    selected_files = valid_df.head(NUM_TEST_FILES).to_dict('records')
    print(f'Selected {len(selected_files)} files for TEST mode')
else:
    selected_files = valid_df.to_dict('records')
    print(f'Selected {len(selected_files)} files for FULL mode')

print(f'Cameras: {sorted(valid_df["camera_type"].unique())}')
print(f'Conditions: {sorted(valid_df["condition"].unique())}')
print(f'Subjects: {valid_df["subject_id"].nunique()}')
display(pd.DataFrame(selected_files)[['subject_id', 'camera_type', 'condition']].head())

In [ ]:
print('Initializing MediaPipe...')
try:
    import mediapipe as mp
    from mediapipe.tasks import python
    from mediapipe.tasks.python import vision
    MEDIAPIPE_AVAILABLE = True
    print('MediaPipe available')
except ImportError as e:
    MEDIAPIPE_AVAILABLE = False
    print(f'MediaPipe not available: {e}')

class TemporalSmoother:
    def __init__(self, window_size=5):
        self.window_size = window_size
        self.history = []
    def smooth(self, landmarks):
        self.history.append(landmarks.copy())
        if len(self.history) > self.window_size:
            self.history.pop(0)
        n = len(self.history)
        weights = [float(i + 1) for i in range(n)]
        smoothed = np.zeros_like(landmarks)
        for i, w in enumerate(weights):
            smoothed += self.history[i] * w
        smoothed /= sum(weights)
        return smoothed

class MediaPipeLandmarkDetector:
    def __init__(self, model_path, smoothing_window=5):
        self.model_path = model_path
        self.smoothing_window = smoothing_window
        self.smoother = TemporalSmoother(smoothing_window)
        self.detector = None
        self.frame_count = 0
        self.fps = 30.0
    def initialize_detector(self):
        if not MEDIAPIPE_AVAILABLE:
            raise RuntimeError('MediaPipe not available')
        base_options = python.BaseOptions(model_asset_path=self.model_path)
        options = vision.FaceLandmarkerOptions(
            base_options=base_options,
            running_mode=vision.RunningMode.VIDEO,
            output_face_blendshapes=True,
            output_facial_transformation_matrixes=True,
            num_faces=1
        )
        self.detector = vision.FaceLandmarker.create_from_options(options)
    def detect_landmarks(self, frame):
        if self.detector is None:
            self.initialize_detector()
        if frame.dtype != np.uint8:
            frame = (frame * 255).astype(np.uint8)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame)
        try:
            timestamp_ms = int(self.frame_count * (1000.0 / self.fps))
            result = self.detector.detect_for_video(mp_image, timestamp_ms)
            self.frame_count += 1
            if result and result.face_landmarks:
                face_landmarks = result.face_landmarks[0]
                frame_width, frame_height = frame.shape[1], frame.shape[0]
                points = np.array([(lm.x * frame_width, lm.y * frame_height) for lm in face_landmarks], dtype='float32')
                if np.any(np.isnan(points)) or np.any(np.isinf(points)):
                    return None
                if np.max(points) > max(frame_width, frame_height) * 3 or np.min(points) < -max(frame_width, frame_height) * 2:
                    return None
                smoothed_points = self.smoother.smooth(points)
                return smoothed_points
            else:
                return None
        except Exception as e:
            print(f'Error: {e}')
            return None
    def reset(self):
        self.frame_count = 0
        self.smoother.history = []
        if self.detector is not None:
            try:
                self.detector.close()
            except Exception as e:
                print(f'Warning: {e}')
            self.detector = None
        self.initialize_detector()

if MEDIAPIPE_AVAILABLE:
    detector = MediaPipeLandmarkDetector(MEDIAPIPE_MODEL_PATH, smoothing_window=SMOOTHING_WINDOW)
    print('MediaPipe detector initialized')
else:
    print('MediaPipe not available')
    detector = None

In [ ]:
def count_video_frames_ffmpeg(video_path):
    try:
        reader = imageio.get_reader(video_path, 'ffmpeg')
        n_frames = reader.count_frames()
        reader.close()
        return n_frames
    except Exception as e:
        print(f'Error: {e}')
        return 0

def get_video_fps(video_path):
    try:
        reader = imageio.get_reader(video_path, 'ffmpeg')
        fps = reader.get_meta_data().get('fps', 30.0)
        reader.close()
        return fps
    except Exception:
        return 30.0

def load_ppg_sync_data(ppg_sync_path):
    try:
        if ppg_sync_path.endswith('.npy'):
            data = np.load(ppg_sync_path)
        elif ppg_sync_path.endswith('.txt'):
            data = np.loadtxt(ppg_sync_path)
        else:
            data = np.load(ppg_sync_path)
        if data.ndim == 1:
            data = data.reshape(-1, 1)
        return data
    except Exception as e:
        print(f'Error: {e}')
        return np.array([])

def create_chunks(n_frames, chunk_size=CHUNK_SIZE, overlap_size=OVERLAP_SIZE):
    chunks = []
    chunk_idx = 0
    start = 0
    while start < n_frames:
        end = min(start + chunk_size, n_frames)
        chunks.append((start, end, chunk_idx))
        if end == n_frames:
            break
        start = end - overlap_size
        chunk_idx += 1
    return chunks

def load_video_frames(video_path, start_frame=0, end_frame=None):
    try:
        reader = imageio.get_reader(video_path, 'ffmpeg')
        total_frames = reader.count_frames()
        if end_frame is None:
            end_frame = total_frames
        frames = []
        for i in range(start_frame, end_frame):
            frames.append(reader.get_next_data())
        reader.close()
        return np.array(frames)
    except Exception as e:
        print(f'Error: {e}')
        return np.array([])

def extract_roi_bbox(landmarks, roi_indices, frame_shape, box_size=ROI_BOX_SIZE):
    valid_indices = [i for i in roi_indices if i < len(landmarks)]
    if not valid_indices:
        return (0, 0, box_size[0], box_size[1])
    roi_points = landmarks[valid_indices]
    raw_x = np.mean(roi_points[:, 0])
    raw_y = np.mean(roi_points[:, 1])
    if not np.isfinite(raw_x) or not np.isfinite(raw_y):
        return (0, 0, box_size[0], box_size[1])
    center_x = max(0, min(int(raw_x), frame_shape[1]))
    center_y = max(0, min(int(raw_y), frame_shape[0]))
    box_w, box_h = box_size
    x = max(0, center_x - box_w // 2)
    y = max(0, center_y - box_h // 2)
    x = min(x, frame_shape[1] - box_w)
    y = min(y, frame_shape[0] - box_h)
    return (int(x), int(y), int(box_w), int(box_h))

def extract_roi_region(frame, bbox):
    x, y, w, h = bbox
    return frame[y:y+h, x:x+w]

print('Utility functions loaded')

In [ ]:
def process_video_chunk(video_path, ppg_sync_path, meta_data, chunk_start, chunk_end, chunk_idx, detector):
    try:
        video_frames = load_video_frames(video_path, chunk_start, chunk_end)
        if len(video_frames) == 0:
            return None
        ppg_sync_data = load_ppg_sync_data(ppg_sync_path)
        if len(ppg_sync_data) == 0:
            return None
        chunk_ppg = ppg_sync_data[chunk_start:chunk_end]
        if chunk_ppg.shape[1] >= 2:
            ppg_values = chunk_ppg[:, 0]
            time_deltas = chunk_ppg[:, 1]
        else:
            ppg_values = chunk_ppg[:, 0] if chunk_ppg.ndim > 1 else chunk_ppg
            time_deltas = np.zeros_like(ppg_values)
        detector.reset()
        detector.fps = get_video_fps(video_path)
        chunk_landmarks = []
        for frame in video_frames:
            lms = detector.detect_landmarks(frame)
            if lms is not None:
                chunk_landmarks.append(lms)
            elif chunk_landmarks:
                chunk_landmarks.append(chunk_landmarks[-1].copy())
            else:
                chunk_landmarks.append(np.zeros((468, 2), dtype='float32'))
        chunk_landmarks = np.array(chunk_landmarks)
        roi_data = {}
        for roi_name, roi_indices in ROIS.items():
            roi_frames = []
            for frame_idx, frame in enumerate(video_frames):
                landmarks = chunk_landmarks[frame_idx]
                if np.all(landmarks == 0):
                    h, w = frame.shape[:2]
                    bbox = (w // 2 - ROI_BOX_SIZE[0] // 2, h // 2 - ROI_BOX_SIZE[1] // 2, ROI_BOX_SIZE[0], ROI_BOX_SIZE[1])
                else:
                    bbox = extract_roi_bbox(landmarks, roi_indices, frame.shape[:2], ROI_BOX_SIZE)
                roi_region = extract_roi_region(frame, bbox)
                roi_frames.append(roi_region)
            roi_data[roi_name] = np.array(roi_frames)
        chunk_meta = {
            'subject_id': meta_data.get('subject_id'),
            'condition': meta_data.get('condition'),
            'camera_type': meta_data.get('camera_type'),
            'view_type': meta_data.get('view_type'),
            'chunk_index': chunk_idx,
            'start_frame': chunk_start,
            'end_frame': chunk_end,
            'num_frames': chunk_end - chunk_start,
        }
        vital_signs = {}
        vital_cols = ['upper_ap', 'lower_ap', 'saturation', 'hemoglobin',
                      'glycated_hemoglobin', 'cholesterol', 'respiratory',
                      'rigidity', 'pulse', 'stress']
        for col in vital_cols:
            if col in meta_data:
                vital_signs[col] = meta_data[col]
        return {
            'roi_data': roi_data,
            'ppg_values': ppg_values,
            'time_deltas': time_deltas,
            'landmarks': chunk_landmarks,
            'metadata': chunk_meta,
            'vital_signs': vital_signs,
        }
    except Exception as e:
        print(f'Error processing chunk: {e}')
        return None

def save_chunk_as_npz(chunk_data, output_path):
    try:
        os.makedirs(output_path, exist_ok=True)
        save_data = {}
        for roi_name, roi_frames in chunk_data['roi_data'].items():
            save_data[f'roi_{roi_name}'] = roi_frames
        save_data['ppg_values'] = chunk_data['ppg_values']
        save_data['time_deltas'] = chunk_data['time_deltas']
        save_data['landmarks'] = chunk_data['landmarks']
        for key, value in chunk_data['metadata'].items():
            save_data[f'meta_{key}'] = value
        for key, value in chunk_data['vital_signs'].items():
            save_data[f'vital_{key}'] = value
        subject_id = chunk_data['metadata']['subject_id']
        camera = chunk_data['metadata']['camera_type']
        condition = chunk_data['metadata']['condition']
        chunk_idx = chunk_data['metadata']['chunk_index']
        filename = f'{subject_id}_{camera}_{condition}_chunk{chunk_idx}.npz'
        filepath = os.path.join(output_path, filename)
        np.savez_compressed(filepath, **save_data)
        return filepath
    except Exception as e:
        print(f'Error saving chunk: {e}')
        return None

print('Chunk processing functions loaded')

In [ ]:
print('Starting main processing...')
processing_stats = {
    'start_time': datetime.now(),
    'total_videos': len(selected_files),
    'total_chunks': 0,
    'total_frames': 0,
    'total_size_mb': 0.0,
    'processing_times': [],
    'camera_counts': {},
    'condition_counts': {},
    'subject_counts': {},
    'chunk_sizes': [],
    'success_count': 0,
    'failure_count': 0,
}

if not MEDIAPIPE_AVAILABLE or detector is None:
    print('ERROR: MediaPipe is not available. Cannot process videos.')
    print('Please install MediaPipe and ensure the model file is available.')
else:
    start_time = time.time()
    all_saved_files = []
    for i, video_row in enumerate(tqdm(selected_files, desc='Processing videos')):
        video_path = video_row['video_full']
        ppg_sync_path = video_row['ppg_sync_full']
        subject_id = video_row['subject_id']
        camera = video_row['camera_type']
        condition = video_row['condition']
        print(f'Video {i+1}/{len(selected_files)}: {subject_id}_{camera}_{condition}')
        meta_data = {
            'subject_id': subject_id,
            'condition': condition,
            'camera_type': camera,
            'view_type': video_row.get('view_type', 'unknown'),
        }
        vital_cols = ['upper_ap', 'lower_ap', 'saturation', 'hemoglobin',
                      'glycated_hemoglobin', 'cholesterol', 'respiratory',
                      'rigidity', 'pulse', 'stress']
        for col in vital_cols:
            if col in video_row:
                meta_data[col] = video_row[col]
        n_frames = count_video_frames_ffmpeg(video_path)
        if n_frames == 0:
            print('No frames found')
            processing_stats['failure_count'] += 1
            continue
        chunks = create_chunks(n_frames, CHUNK_SIZE, OVERLAP_SIZE)
        print(f'Frames: {n_frames}, Chunks: {len(chunks)}')
        video_start_time = time.time()
        video_saved_files = []
        for start, end, chunk_idx in chunks:
            chunk_data = process_video_chunk(
                video_path, ppg_sync_path, meta_data, start, end, chunk_idx, detector
            )
            if chunk_data is not None:
                filepath = save_chunk_as_npz(chunk_data, CHUNKS_PATH)
                if filepath:
                    video_saved_files.append(filepath)
                    file_size = os.path.getsize(filepath) / (1024 * 1024)
                    processing_stats['total_size_mb'] += file_size
                    processing_stats['total_chunks'] += 1
                    processing_stats['total_frames'] += chunk_data['metadata']['num_frames']
                    processing_stats['chunk_sizes'].append(chunk_data['metadata']['num_frames'])
                    processing_stats['camera_counts'][camera] = processing_stats['camera_counts'].get(camera, 0) + 1
                    processing_stats['condition_counts'][condition] = processing_stats['condition_counts'].get(condition, 0) + 1
                    processing_stats['subject_counts'][subject_id] = processing_stats['subject_counts'].get(subject_id, 0) + 1
        video_time = time.time() - video_start_time
        processing_stats['processing_times'].append(video_time)
        all_saved_files.extend(video_saved_files)
        if len(video_saved_files) > 0:
            processing_stats['success_count'] += 1
            print(f'Saved {len(video_saved_files)} chunks')
        else:
            processing_stats['failure_count'] += 1
            print('No chunks saved')
        gc.collect()
    processing_stats['end_time'] = datetime.now()
    processing_stats['total_processing_time'] = time.time() - start_time
    print('Processing complete')

In [ ]:
print('=' * 80)
print('COMPREHENSIVE STATISTICS')
print('=' * 80)
print()
print('OVERVIEW')
print('-' * 60)
print(f'Processing Mode: {PROCESSING_MODE}')
print(f'Start Time: {processing_stats["start_time"]}')
print(f'End Time: {processing_stats["end_time"]}')
print(f'Total Processing Time: {processing_stats["total_processing_time"]:.2f} seconds')
print()
print('PROCESSING SUMMARY')
print('-' * 60)
print(f'Total Videos Processed: {processing_stats["total_videos"]}')
print(f'Successful Videos: {processing_stats["success_count"]}')
print(f'Failed Videos: {processing_stats["failure_count"]}')
if processing_stats["total_videos"] > 0:
    print(f'Success Rate: {(processing_stats["success_count"] / processing_stats["total_videos"] * 100):.1f}%')
print()
print('CHUNK STATISTICS')
print('-' * 60)
print(f'Total Chunks Saved: {processing_stats["total_chunks"]}')
print(f'Total Frames Processed: {processing_stats["total_frames"]}')
if processing_stats["success_count"] > 0:
    print(f'Average Frames per Video: {processing_stats["total_frames"] / processing_stats["success_count"]:.1f}')
    print(f'Average Chunks per Video: {processing_stats["total_chunks"] / processing_stats["success_count"]:.1f}')
print()
if processing_stats['chunk_sizes']:
    chunk_sizes_array = np.array(processing_stats['chunk_sizes'])
    print('Chunk Size Statistics:')
    print(f'  Min: {chunk_sizes_array.min()} frames')
    print(f'  Max: {chunk_sizes_array.max()} frames')
    print(f'  Mean: {chunk_sizes_array.mean():.1f} frames')
    print(f'  Median: {np.median(chunk_sizes_array):.1f} frames')
    print()
print('CAMERA DISTRIBUTION')
print('-' * 60)
camera_df = pd.DataFrame(list(processing_stats['camera_counts'].items()), columns=['Camera', 'Chunks'])
camera_df = camera_df.sort_values('Chunks', ascending=False)
print(camera_df.to_string(index=False))
print()
plt.figure(figsize=(10, 6))
camera_df.plot(kind='bar', x='Camera', y='Chunks', color='skyblue', legend=False)
plt.title('Chunks Processed by Camera Type')
plt.xlabel('Camera Type')
plt.ylabel('Number of Chunks')
plt.xticks(rotation=45)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()
print()
print('CONDITION DISTRIBUTION')
print('-' * 60)
condition_df = pd.DataFrame(list(processing_stats['condition_counts'].items()), columns=['Condition', 'Chunks'])
condition_df = condition_df.sort_values('Chunks', ascending=False)
print(condition_df.to_string(index=False))
print()
plt.figure(figsize=(8, 6))
condition_df.plot(kind='bar', x='Condition', y='Chunks', color='lightgreen', legend=False)
plt.title('Chunks Processed by Condition')
plt.xlabel('Condition')
plt.ylabel('Number of Chunks')
plt.xticks(rotation=45)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()
print()
print('SUBJECT DISTRIBUTION')
print('-' * 60)
subject_df = pd.DataFrame(list(processing_stats['subject_counts'].items()), columns=['Subject', 'Chunks'])
subject_df = subject_df.sort_values('Chunks', ascending=False)
print(f'Total Unique Subjects: {len(subject_df)}')
print(f'Top 10 Subjects by Chunks:')
print(subject_df.head(10).to_string(index=False))
print()
print('PROCESSING TIME STATISTICS')
print('-' * 60)
if processing_stats['processing_times']:
    times_array = np.array(processing_stats['processing_times'])
    print(f'Total Processing Time: {processing_stats["total_processing_time"]:.2f} seconds')
    print(f'Average Time per Video: {times_array.mean():.2f} seconds')
    print(f'Min Time: {times_array.min():.2f} seconds')
    print(f'Max Time: {times_array.max():.2f} seconds')
    print(f'Median Time: {np.median(times_array):.2f} seconds')
    print()
    plt.figure(figsize=(10, 6))
    plt.plot(times_array, marker='o', linestyle='-', color='purple')
    plt.title('Processing Time per Video')
    plt.xlabel('Video Index')
    plt.ylabel('Processing Time (seconds)')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    print()
print('STORAGE STATISTICS')
print('-' * 60)
print(f'Total Output Size: {processing_stats["total_size_mb"]:.2f} MB')
if processing_stats["total_chunks"] > 0:
    print(f'Average Chunk Size: {processing_stats["total_size_mb"] / processing_stats["total_chunks"]:.2f} MB')
print()
print('ROI STATISTICS')
print('-' * 60)
print(f'Number of ROIs per Chunk: {len(ROIS)}')
print(f'ROI Box Size: {ROI_BOX_SIZE}')
print(f'ROI Names: {list(ROIS.keys())}')
print()
if PROCESSING_MODE == 'test':
    print('ESTIMATED FULL DATASET STATISTICS')
    print('-' * 60)
    total_videos_in_dataset = len(valid_df)
    avg_chunks_per_video = processing_stats['total_chunks'] / processing_stats['success_count']
    avg_chunk_size_mb = processing_stats['total_size_mb'] / processing_stats['total_chunks']
    avg_time_per_video = processing_stats['total_processing_time'] / processing_stats['success_count']
    estimated_total_chunks = int(total_videos_in_dataset * avg_chunks_per_video)
    estimated_total_size_gb = (total_videos_in_dataset * avg_chunk_size_mb) / 1024
    estimated_total_time_hours = (total_videos_in_dataset * avg_time_per_video) / 3600
    print(f'Dataset Size: {total_videos_in_dataset} videos')
    print(f'Estimated Total Chunks: {estimated_total_chunks}')
    print(f'Estimated Total Size: {estimated_total_size_gb:.2f} GB')
    print(f'Estimated Total Time: {estimated_total_time_hours:.2f} hours')
    print()
print('SAVED FILES')
print('-' * 60)
if all_saved_files:
    print(f'Total Files Saved: {len(all_saved_files)}')
    print('First 10 files:')
    for i, filepath in enumerate(all_saved_files[:10]):
        size_mb = os.path.getsize(filepath) / (1024 * 1024)
        print(f'  {i+1}. {os.path.basename(filepath)} ({size_mb:.2f} MB)')
    print()
print('=' * 80)
print('FINAL SUMMARY')
print('=' * 80)
print(f'Processing Mode: {PROCESSING_MODE}')
print(f'Videos Processed: {processing_stats["success_count"]}/{processing_stats["total_videos"]}')
print(f'Total Chunks: {processing_stats["total_chunks"]}')
print(f'Total Frames: {processing_stats["total_frames"]}')
print(f'Total Size: {processing_stats["total_size_mb"]:.2f} MB')
print(f'Processing Time: {processing_stats["total_processing_time"]:.2f} seconds')
if processing_stats["total_videos"] > 0:
    print(f'Success Rate: {(processing_stats["success_count"] / processing_stats["total_videos"] * 100):.1f}%')
print()
print('=' * 80)
print('ALL DONE!')
print('=' * 80)

In [ ]:
print('Saving statistics...')
statistics = {
    'processing_info': {
        'mode': PROCESSING_MODE,
        'start_time': processing_stats['start_time'].isoformat(),
        'end_time': processing_stats['end_time'].isoformat(),
        'total_processing_time_seconds': processing_stats['total_processing_time'],
    },
    'processing_summary': {
        'total_videos': processing_stats['total_videos'],
        'success_count': processing_stats['success_count'],
        'failure_count': processing_stats['failure_count'],
        'success_rate_percent': (processing_stats['success_count'] / processing_stats['total_videos'] * 100),
    },
    'chunk_statistics': {
        'total_chunks': processing_stats['total_chunks'],
        'total_frames': processing_stats['total_frames'],
        'avg_frames_per_video': processing_stats['total_frames'] / processing_stats['success_count'] if processing_stats['success_count'] > 0 else 0,
        'avg_chunks_per_video': processing_stats['total_chunks'] / processing_stats['success_count'] if processing_stats['success_count'] > 0 else 0,
    },
    'distribution': {
        'camera_counts': processing_stats['camera_counts'],
        'condition_counts': processing_stats['condition_counts'],
        'subject_counts': processing_stats['subject_counts'],
    },
    'storage': {
        'total_size_mb': processing_stats['total_size_mb'],
        'avg_chunk_size_mb': processing_stats['total_size_mb'] / processing_stats['total_chunks'] if processing_stats['total_chunks'] > 0 else 0,
    },
    'configuration': {
        'chunk_size': CHUNK_SIZE,
        'overlap_size': OVERLAP_SIZE,
        'roi_box_size': list(ROI_BOX_SIZE),
        'smoothing_window': SMOOTHING_WINDOW,
    },
    'roi_info': {
        'num_rois': len(ROIS),
        'roi_names': list(ROIS.keys()),
    }
}
stats_file = os.path.join(OUTPUT_PATH, 'processing_statistics.json')
with open(stats_file, 'w') as f:
    json.dump(statistics, f, indent=2, default=str)
print(f'Statistics saved to: {stats_file}')
stats_summary = {
    'metric': ['Processing Mode', 'Total Videos', 'Success Count', 'Failure Count',
               'Success Rate (%)', 'Total Chunks', 'Total Frames',
               'Total Size (MB)', 'Processing Time (s)'],
    'value': [
        PROCESSING_MODE,
        processing_stats['total_videos'],
        processing_stats['success_count'],
        processing_stats['failure_count'],
        (processing_stats['success_count'] / processing_stats['total_videos'] * 100),
        processing_stats['total_chunks'],
        processing_stats['total_frames'],
        processing_stats['total_size_mb'],
        processing_stats['total_processing_time']
    ]
}
stats_summary_df = pd.DataFrame(stats_summary)
stats_csv_file = os.path.join(OUTPUT_PATH, 'processing_summary.csv')
stats_summary_df.to_csv(stats_csv_file, index=False)
print(f'Summary CSV saved to: {stats_csv_file}')
print('Statistics saved successfully')